# 🎯 DipRadar — Bootstrap ML (Colab)

Corre cada célula **uma de cada vez**, por ordem.
O Parquet acumula no Google Drive — podes fechar o Colab entre sessões sem perder progresso.

**Fase A — CamadaA (Price + Macro):**

| Passo | O que faz | Tempo estimado |
|-------|-----------|----------------|
| Batch 0–200 | Macro bulk fetch (3 pedidos) + backfill preços + features | ~25 min |
| Batch 200–400 | idem (macro já em cache no Drive) | ~25 min |
| Batch 400–600 | idem | ~25 min |
| Batch 600–800 | idem | ~25 min |
| Batch 800–fim | idem | ~10 min |

**Fase B — CamadaB (Fundamentais PIT):**

| Passo | O que faz | Tempo estimado |
|-------|-----------|----------------|
| Fund 0–200 | quarterly_income_stmt + balance_sheet + cashflow PIT | ~35 min |
| Fund 200–400 | idem | ~35 min |
| Fund 400–600 | idem | ~35 min |
| Fund 600–fim | idem | ~25 min |
| Treino Ensemble (RF/XGBoost/LightGBM autoselect) | train_model.py (16 features) | ~5 min |
| **Total** | | **~4 h 30 min** |

> ⚠️ **Cada sessão nova do Colab**: volta a correr as células 1, 2, 3 e 4 antes de continuar.

> ℹ️ O download macro (VIX, SPY, T10Y2Y) é feito automaticamente no **primeiro batch de cada sessão** — não há passo separado.

> 🛡️ **CamadaB usa Point-in-Time rigoroso**: reporting lag de 45 dias + fallback NaN (nunca `tk.info`). Se um batch falhar por Rate Limit, volta a correr a célula — os tickers já processados são saltados.

## 1 · Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## 2 · Instalar dependências

In [ ]:
%pip install -q yfinance scikit-learn pyarrow fastparquet xgboost lightgbm shap pandas-datareader
print('✅ Dependências instaladas')

## 3 · Clonar / actualizar repositório

In [ ]:
import os

REPO_DIR = '/content/DipRadar'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull --quiet
    print('✅ Repositório actualizado')
else:
    !git clone --quiet https://github.com/romeurf/DipRadar.git $REPO_DIR
    %cd $REPO_DIR
    print('✅ Repositório clonado')

DRIVE_DIR = '/content/drive/MyDrive/DipRadar'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'📁 Drive dir: {DRIVE_DIR}')

## 4 · Configurar API Key (Tiingo)

> Guarda a chave em: **Colab → Secrets** (ícone da chave 🔑) → Nome: `TIINGO_API_KEY`

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['TIINGO_API_KEY'] = userdata.get('TIINGO_API_KEY')
    print('✅ TIINGO_API_KEY carregada dos Secrets do Colab')
except Exception:
    import getpass
    os.environ['TIINGO_API_KEY'] = getpass.getpass('🔑 Cola a tua TIINGO_API_KEY: ')
    print('✅ TIINGO_API_KEY definida via input')

---
## ⚡ FASE A — CamadaA: Preços + Macro
---

## 5 · Batch 1 — tickers 0–200

> O `build_global_macro_df()` corre automaticamente no início deste batch (3 pedidos de rede, ~3 s).  
> Se a sessão expirar a meio, volta a correr — os tickers já processados são saltados.

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 6 · Batch 2 — tickers 200–400

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 7 · Batch 3 — tickers 400–600

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 8 · Batch 4 — tickers 600–800

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 600 800 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 9 · Batch 5 — tickers 800 até ao fim

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 800 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

---
## 🧮 FASE B — CamadaB: Fundamentais Point-in-Time

> Enriquece cada alerta do Parquet com `gross_margin`, `fcf_yield`, `revenue_growth` e `de_ratio`  
> extraídos dos `quarterly_income_stmt` / `balance_sheet` / `cashflow` **já públicos na data do dip**  
> (reporting lag de 45 dias). Fallback é **NaN** — nunca `tk.info`.
---

## 10 · Fund Batch 1 — tickers 0–200 (US Core)

> ⏱️ ~35 min. Faz 3 pedidos de rede extra por ticker (quarterly statements).  
> Se o Yahoo Finance devolver Rate Limit, o Colab irá abrandar — é normal. Volta a correr se necessário.

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 11 · Fund Batch 2 — tickers 200–400

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 12 · Fund Batch 3 — tickers 400–600

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 13 · Fund Batch 4 — tickers 600–fim (Europa / Edge Cases)

In [ ]:
!python bootstrap_ml.py \
    --layer fund \
    --slice 600 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

---
## 🏋️ FASE C — Treino do Ensemble Completo
---

## 14 · Merge Camada A + Camada B + Treino

> Só depois de todos os batches (Fase A + Fase B) estarem completos.  
> O `train_model.py` faz autoselect entre RF / XGBoost / LightGBM e guarda o vencedor.  
> Gera `dip_model_stage1.pkl`, `dip_model_stage2.pkl` e `ml_report.json`.

In [ ]:
# --- Merge do Super-Dataset (Camada A + Camada B) ---
import pandas as pd

print("A carregar Parquets do Google Drive...")
df_price = pd.read_parquet("/content/drive/MyDrive/DipRadar/ml_training_price.parquet")
df_fund  = pd.read_parquet("/content/drive/MyDrive/DipRadar/ml_training_fund.parquet")

print(f"📊 Amostras Camada A (Price+Macro): {len(df_price)}")
print(f"📊 Amostras Camada B (Fundamentais PIT): {len(df_fund)}")

# Marcar a origem de cada linha
df_price["_source"] = "price"
df_fund["_source"]  = "fund"

# Combinar — fund fica por último para que keep='last' prefira PIT reais
df_merged = pd.concat([df_price, df_fund], ignore_index=True)

# Deduplicar por symbol + alert_date (colunas correctas do bootstrap_ml.py)
antes = len(df_merged)
df_merged = df_merged.drop_duplicates(subset=["symbol", "alert_date"], keep="last")
print(f"🔁 Deduplicação: {antes} → {len(df_merged)} alertas únicos")

# Remover coluna auxiliar antes de passar ao train_model
df_merged = df_merged.drop(columns=["_source"], errors="ignore")

print(f"🚀 Total combinado final: {len(df_merged)} alertas")
print(f"📈 Distribuição de outcomes:\n{df_merged['outcome_label'].value_counts()}")

# Verificar NaNs nas features críticas
from ml_features import FEATURE_COLUMNS
nan_summary = df_merged[FEATURE_COLUMNS].isna().sum()
nan_cols = nan_summary[nan_summary > 0]
if not nan_cols.empty:
    print(f"⚠️  NaNs detectados (serão imputados pelo train_model):\n{nan_cols}")
else:
    print("✅ Sem NaNs nas features")

merged_path = "/content/drive/MyDrive/DipRadar/ml_training_merged.parquet"
df_merged.to_parquet(merged_path)
print(f"✅ Merge concluído e guardado em: {merged_path}")

In [ ]:
# --- CÉLULA FINAL: Treino do Ensemble Completo ---
!python train_model.py \
    --parquet /content/drive/MyDrive/DipRadar/ml_training_merged.parquet \
    --output-dir /content/drive/MyDrive/DipRadar

## 15 · Validar contrato de features (anti Training-Serving Skew)

> Confirma que os `.pkl` treinados e o `ml_features.py` em produção têm **exactamente as mesmas features**, pela mesma ordem.  
> **Não saltes este passo** — é o guarda que impede um crash silencioso em produção.

In [ ]:
import pickle, pathlib
from ml_features import FEATURE_COLUMNS, N_FEATURES

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')

# Os modelos gerados pelo train_model.py são stage1 e stage2
for pkl_name in ['dip_model_stage1.pkl', 'dip_model_stage2.pkl']:
    pkl_path = drive_dir / pkl_name
    if not pkl_path.exists():
        print(f'⚠️  {pkl_name} não encontrado — corre a Fase C primeiro')
        continue

    with open(pkl_path, 'rb') as f:
        bundle = pickle.load(f)

    # O bundle guarda feature_columns explicitamente (contrato do train_model.py)
    model_features = list(bundle.get('feature_columns', []))
    code_features  = list(FEATURE_COLUMNS)

    if model_features == code_features:
        print(f'✅ {pkl_name} — contrato OK ({N_FEATURES} features)')
    else:
        missing = set(code_features) - set(model_features)
        extra   = set(model_features) - set(code_features)
        print(f'❌ {pkl_name} — MISMATCH detectado (Training-Serving Skew)!')
        if missing: print(f'  Faltam no modelo : {missing}')
        if extra:   print(f'  Extra no modelo  : {extra}')
        raise AssertionError(f'{pkl_name}: corrige antes de fazer deploy.')

print('\nFeatures em produção:')
for i, f in enumerate(FEATURE_COLUMNS):
    print(f'  {i+1:>2}. {f}')

## 16 · Verificar todos os ficheiros gerados

In [ ]:
import os, pathlib, pandas as pd

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')
print('📁 Ficheiros no Drive:')
for f in sorted(drive_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>8.1f} KB')

merged_pq = drive_dir / 'ml_training_merged.parquet'
if merged_pq.exists():
    df = pd.read_parquet(merged_pq)
    print(f'\n📊 Merged Parquet: {len(df):,} alertas | {df["symbol"].nunique()} tickers únicos')
    print('\nDistribuição outcome_label:')
    print(df['outcome_label'].value_counts().to_string())
    fund_cols = ['gross_margin', 'fcf_yield', 'revenue_growth', 'de_ratio']
    existing = [c for c in fund_cols if c in df.columns]
    if existing:
        print('\n🧮 Cobertura dos Fundamentais PIT (% não-NaN):')
        for col in existing:
            pct = df[col].notna().mean() * 100
            print(f'  {col:<20} {pct:>5.1f}% preenchido')
else:
    print('⚠️  Merged Parquet ainda não existe — corre a Fase C primeiro.')

## 17 · Deploy do modelo para o Railway

### Opção A — Railway Volume (recomendado)
O Railway tem um Volume montado em `/data`.
Descarrega os `.pkl` do Drive para a tua máquina local e faz upload via Railway CLI:
```bash
railway volume cp ./dip_model_stage1.pkl /data/dip_model_stage1.pkl
railway volume cp ./dip_model_stage2.pkl /data/dip_model_stage2.pkl
```

### Opção B — Commit directo (só se < 50 MB por ficheiro)
```bash
git add dip_model_stage1.pkl dip_model_stage2.pkl
git commit -m 'feat: add trained ML models v2'
git push
```